In [22]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2382.50it/s]


STEP_1: Indexing (Document Ingestion)

In [23]:
from youtube_transcript_api._errors import TranscriptsDisabled

In [24]:
video_id = "P9OC4n9QOxU"

api = YouTubeTranscriptApi()

try:
    transcript_list = api.fetch(video_id, languages=["en"])

    # Convert the transcript into plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

Why do things stick in your mind? Because they have some substance. >> Power. That's why they stay >> in your mind. Someone said it directly, someone said it indirectly, someone said it there, saying, " >> You guys haven't written OTT, you don't know that word. If I didn't cry or Raju didn't cry, the world wouldn't cry watching that scene. " Sometimes, I feel very frustrated, thinking, " >> This is over, it's not happening anymore ." Four friends went on a picnic and came back. Those four friends went, and only three returned. The demands of the target audience for writers have changed. >> Our today's game-changing writers have such talent that their films are full of entertainment, and because of that rich entertainment, their films bring in such a steady stream of money at the box office. But another thing is that their films have such a social message that, from the first film to the last, their social message is remembered for years. Hello, hello, when we talk about your films, the

In [25]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Why do things stick in your mind?', start=9.099, duration=3.143), FetchedTranscriptSnippet(text='Because they have some substance.', start=10.702, duration=2.838), FetchedTranscriptSnippet(text='>> Power.', start=12.242, duration=2.362), FetchedTranscriptSnippet(text="That's why they stay", start=13.54, duration=3.22), FetchedTranscriptSnippet(text='>> in your mind. Someone said it directly,', start=14.604, duration=4.286), FetchedTranscriptSnippet(text='someone said it indirectly, someone said', start=16.76, duration=3.218), FetchedTranscriptSnippet(text='it there, saying, "', start=18.89, duration=2.786), FetchedTranscriptSnippet(text=">> You guys haven't written OTT, you don't", start=19.978, duration=4.336), FetchedTranscriptSnippet(text='know that word.', start=21.676, duration=2.638), FetchedTranscriptSnippet(text="If I didn't cry or Raju didn't cry, the", start=27.787, duration=5.093), FetchedTranscriptSnippet(text="worl

STEP_1: INDEXING (Text Splitting)

In [26]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [27]:
len(chunks)

51

In [28]:
chunks[0]

Document(metadata={}, page_content='Why do things stick in your mind? Because they have some substance. >> Power. That\'s why they stay >> in your mind. Someone said it directly, someone said it indirectly, someone said it there, saying, " >> You guys haven\'t written OTT, you don\'t know that word. If I didn\'t cry or Raju didn\'t cry, the world wouldn\'t cry watching that scene. " Sometimes, I feel very frustrated, thinking, " >> This is over, it\'s not happening anymore ." Four friends went on a picnic and came back. Those four friends went, and only three returned. The demands of the target audience for writers have changed. >> Our today\'s game-changing writers have such talent that their films are full of entertainment, and because of that rich entertainment, their films bring in such a steady stream of money at the box office. But another thing is that their films have such a social message that, from the first film to the last, their social message is remembered for years. Hell

STEP_1 : INDEXING (Embedding Generation and Storing in vector store)

In [29]:
vector_store = FAISS.from_documents(chunks,embedding_model)

In [30]:
vector_store.index_to_docstore_id

{0: '25c810fc-b2a0-490a-a4de-a027eae6409d',
 1: 'b94967cc-2a71-407d-9923-95b52d681c4f',
 2: 'b1d7b613-88e4-4749-bd04-7d68ac8ed3c6',
 3: 'f86fe4ee-c3e8-4e25-aa18-0e738d8c377b',
 4: 'd90da172-ab2f-4e0d-b516-7f1345ea2bbe',
 5: '49e384fa-a53a-40c0-9b9c-dd7cf8facc11',
 6: '6da113d8-d88c-4af7-97cc-11bcc0805bb0',
 7: 'bcec1d53-c262-48ff-9cc9-5f0040d57537',
 8: '392f2785-a89d-4783-8b47-85a552aab614',
 9: '965382e5-6cc4-47e9-a4c2-bdbb6a4e2b06',
 10: '4ddf98e7-274d-4233-92d7-f0b8fa05db27',
 11: '4f555820-2490-4450-bffa-46a2ee2b51e5',
 12: '0aae49c8-22df-42f5-874c-e54f73519595',
 13: '46b76673-4e12-40d0-b19b-4ea51b5e8b1b',
 14: '9e3b9e53-ebc9-46d5-896c-ed2866666f19',
 15: '25b92517-8547-4ff2-8db0-7aa5ae211fc0',
 16: 'b6f531c5-12a8-4f16-a71b-67e5c697d829',
 17: 'f31e1a03-8dbf-466d-b5bb-4faa6daf557a',
 18: '60cb1c39-091e-45e3-bb75-8c46b5457d95',
 19: '3f29d6ab-d788-4d9b-8aa7-a01139ec224b',
 20: '8bad525c-4325-4019-ae89-f1a80bf3d770',
 21: '39e9ca0c-4d7f-4d35-b435-308c73498275',
 22: 'ee72f06a-92a7-

In [42]:
vector_store.get_by_ids(['22b49091-cd87-4118-9bc3-8328cc3219b4'])

[Document(id='22b49091-cd87-4118-9bc3-8328cc3219b4', metadata={}, page_content='in "Three Days" when I didn\'t want to become an engineer. >> Right? >> For his hobby, he bought a laptop for it , and Raju Rastogi interviews the panel . I had written this scene for 15-20 days . I went there, thinking it was all right . And then I went to Vinod Chopra\'s Santa Cruz office. He\'s at Vinod Chopra\'s Santa Cruz office . I gave it to him. So I asked him , "I wrote it. So nothing happened here. Nothing happened here. Now it\'s important that it happens." So I left. I thought , "If I don\'t cry, or Raju doesn\'t cry, the world won\'t cry watching that scene . Or if we\'re not acting the part, if we\'re not laughing so hard, then they\'re not going to laugh. It\'s craft. >> But it\'s not just craft. >> What happens, like the scene where Rajat is just saying, \'We had to show that Sharman is scared throughout this. Otherwise, what will happen to my family ?\' And Raju teaches him to get rid of hi

STEP_2: RETRIEVAL

In [32]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs={"k":4})

In [33]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E8ACE42D50>, search_kwargs={'k': 4})

In [34]:
retriever.invoke("what is the image of the future")

[Document(id='d90da172-ab2f-4e0d-b516-7f1345ea2bbe', metadata={}, page_content='and I feel like this film is a little short, that\'s the case. Otherwise, it\'s not about conveying a message. That\'s not the case. Rather, Raju says, "Don\'t give so much wisdom. Don\'t give so much wisdom." >> Actually, today\'s audience doesn\'t want that kind of wisdom. Especially the youth today, the millennials or Gen Z, as we call them, " I think the world has changed over the years." The demands of the target audience for writers have changed. >> We were told this, but then when we wrote "Pritam Petro," and we wrote it with great hesitation, thinking the world has changed, we realized that the world hasn\'t. Because a human being remains a human being. Whatever he wants. >> But evolution also happens. It does >> evolve. >> Yes. It does >> evolve. So, your conviction connects with that. >> Hmm. >> I think when you try to second-guess what people like today? What will happen ? >> Hmm. Hmm, that\'s wh

STEP_3: AUGMENTATION

In [35]:
llm = ChatGroq(model = "openai/gpt-oss-120b" , temperature=0.2)

In [36]:
prompt = PromptTemplate(
    template=""" 
you are a helpful assistant.
Answer only from the provided transcript context.
If the context is insufficient, just say you don;t know.

{context}
Question: {question}
""",
input_variables=['context','question']
)

In [37]:
question = "is the topic of screenwriting discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [38]:
retrieved_docs

[Document(id='4ddf98e7-274d-4233-92d7-f0b8fa05db27', metadata={}, page_content="The picture will become boring. >> Hmm. >> So, how do you show that and what we call exposition, to reveal that he's honest, but also to >> not be boring. Hmm. There's a >> lot of craft involved in concealing exposition. Screenwriting requires a lot of effort. Theater is different. Theater is dialogue-based. Whatever you want to say, you have to imagine it. You have to imagine a train leaving from here. You have to convey it through dialogue. Screenwriting is a very different craft because you're seeing so many things. If you want to know a backstory, you have to learn it somewhere. You either learn it by watching movies or reading books. >> And I remember our evolution over the years, learning techniques like in Lage Raho Munna Bhai. I'll give you concrete examples. >> Hmm hmm, absolutely. There was >> a moment in Lage Raho Munna Bhai when Munna writes a letter, a note to Lucky. We wanted it to be so insul

In [39]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'The picture will become boring. >> Hmm. >> So, how do you show that and what we call exposition, to reveal that he\'s honest, but also to >> not be boring. Hmm. There\'s a >> lot of craft involved in concealing exposition. Screenwriting requires a lot of effort. Theater is different. Theater is dialogue-based. Whatever you want to say, you have to imagine it. You have to imagine a train leaving from here. You have to convey it through dialogue. Screenwriting is a very different craft because you\'re seeing so many things. If you want to know a backstory, you have to learn it somewhere. You either learn it by watching movies or reading books. >> And I remember our evolution over the years, learning techniques like in Lage Raho Munna Bhai. I\'ll give you concrete examples. >> Hmm hmm, absolutely. There was >> a moment in Lage Raho Munna Bhai when Munna writes a letter, a note to Lucky. We wanted it to be so insulting that we tried it for a long time. Then suddenly one day it occurred to

In [40]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [41]:
final_prompt

StringPromptValue(text=' \nyou are a helpful assistant.\nAnswer only from the provided transcript context.\nIf the context is insufficient, just say you don;t know.\n\nThe picture will become boring. >> Hmm. >> So, how do you show that and what we call exposition, to reveal that he\'s honest, but also to >> not be boring. Hmm. There\'s a >> lot of craft involved in concealing exposition. Screenwriting requires a lot of effort. Theater is different. Theater is dialogue-based. Whatever you want to say, you have to imagine it. You have to imagine a train leaving from here. You have to convey it through dialogue. Screenwriting is a very different craft because you\'re seeing so many things. If you want to know a backstory, you have to learn it somewhere. You either learn it by watching movies or reading books. >> And I remember our evolution over the years, learning techniques like in Lage Raho Munna Bhai. I\'ll give you concrete examples. >> Hmm hmm, absolutely. There was >> a moment in L

STEP_4 : GENERATION

In [43]:
answer = llm.invoke(final_prompt)
print(answer)

content='Yes. The speakers spend a good portion of the conversation talking about **screenwriting**. They discuss:\n\n* How screenwriting differs from theater and novel writing.  \n* The challenge of delivering exposition (e.g., showing a character is honest) without making the scene boring.  \n* Techniques for “show, not tell” – using actions, dialogue, and visual moments to reveal traits and back‑story.  \n* The craft of concealing exposition and deciding what to include in a script.  \n* How writers learn the craft—reading books, watching films, developing personal methods rather than formal classes.  \n* The balance between entertaining the audience and embedding a social or thematic message.  \n\nSo the video does cover screenwriting, focusing on its unique challenges and practical approaches.' additional_kwargs={'reasoning_content': 'We need to answer based only on provided transcript. The question: "is the topic of screenwriting discussed in this video? if yes then what was disc

BUILDING A CHAIN

In [44]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [45]:
def format_docs(retrieved_docs):
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [46]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [47]:
parallel_chain.invoke('who is raju')

{'context': 'The picture will become boring. >> Hmm. >> So, how do you show that and what we call exposition, to reveal that he\'s honest, but also to >> not be boring. Hmm. There\'s a >> lot of craft involved in concealing exposition. Screenwriting requires a lot of effort. Theater is different. Theater is dialogue-based. Whatever you want to say, you have to imagine it. You have to imagine a train leaving from here. You have to convey it through dialogue. Screenwriting is a very different craft because you\'re seeing so many things. If you want to know a backstory, you have to learn it somewhere. You either learn it by watching movies or reading books. >> And I remember our evolution over the years, learning techniques like in Lage Raho Munna Bhai. I\'ll give you concrete examples. >> Hmm hmm, absolutely. There was >> a moment in Lage Raho Munna Bhai when Munna writes a letter, a note to Lucky. We wanted it to be so insulting that we tried it for a long time. Then suddenly one day it

In [48]:
parser = StrOutputParser()

In [49]:
main_chain = parallel_chain | prompt | llm | parser

In [50]:
main_chain.invoke("can you summarize the video")

'**Summary of the video (based on the transcript):**\n\nThe speakers discuss the challenges and craft of screenwriting, especially how to convey character traits and back‑story without resorting to boring exposition. They contrast theater—where dialogue must carry all information—with film, which can show actions visually.  \n\nKey points include:\n\n* **Showing, not telling:** In film you must create scenes that demonstrate traits (e.g., a character’s honesty) through actions rather than simply stating them, otherwise the picture becomes dull.  \n* **Crafting exposition:** Good writers hide exposition within engaging moments, using subtext, humor, or surprising details (e.g., the anecdote about Munna’s insulting note in *Lage Raho Munna\u202fBhai*).  \n* **Iterative writing process:** Scenes are often rewritten or discarded; the first lines may change dramatically as the story evolves.  \n* **Learning the trade:** Successful screenwriters typically haven’t taken formal classes; they r